# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs using Croissant schema.

Below we enumerate all record sets and their fields, referencing entities by their `@id` values as per FAIR^2 principles.

In [ ]:
# List record sets and their fields
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
else:
    # fallback: get record sets via dataset.record_sets if possible
    record_sets = list(dataset.record_sets())

for record_set in record_sets:
    print(f"RecordSet @id: {record_set['@id']} | Name: {record_set.get('name', '')}")
    # List fields in the record set
    fields = record_set.get('field', [])
    for field in fields:
        print(f"  Field @id: {field['@id']} | Name: {field.get('name', '')} | DataType: {field.get('dataType', '')}")
    # List columns if present
    columns = record_set.get('column', [])
    for column in columns:
        print(f"  Column @id: {column['@id']} | Name: {column.get('name', '')} | DataType: {column.get('dataType', '')}")

## 3. Data Extraction
Load data from each specific record set into a DataFrame for analysis.

Below, we extract the data for each available record set, using their `@id`s. Modify the list to use your dataset's actual record set IDs.

In [ ]:
# Dynamically collect record set @id values
record_set_ids = []
if record_sets:
    for record_set in record_sets:
        record_set_ids.append(record_set['@id'])
else:
    # fallback: get record sets from dataset (as strings)
    record_set_ids = list(dataset.record_sets())

dataframes = {}
for record_set_id in record_set_ids:
    # Load records from this record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Could not load RecordSet '@id': {record_set_id} ({e})")

# Choose one record set to preview its data
if dataframes:
    preview_record_set_id = list(dataframes.keys())[0]
    print(f"Preview from RecordSet '@id': {preview_record_set_id}")
    display(dataframes[preview_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, categorization, and grouping using the record set and field `@id`s.

Below, we'll demonstrate with a numeric and categorical field. Please update the `numeric_field_id` and `group_field_id` to match your actual dataset's schema as shown in the overview.

In [ ]:
# Example EDA: Using one record set
from IPython.display import display

if dataframes:
    record_set_id = preview_record_set_id  # Use the previously selected record set
    df = dataframes[record_set_id]

    # Identify numeric columns (as candidates for numeric field)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Pick first numeric for demo
        print(f"Using Numeric Field '@id': {numeric_field_id}")

        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping by a categorical field
        cat_cols = df.select_dtypes(exclude=['number']).columns.tolist()
        if cat_cols:
            group_field_id = cat_cols[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print(f"Categorical field {group_field_id} not in filtered_df columns.")
        else:
            print('No categorical columns found for grouping.')
    else:
        print("No numeric fields found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields using pandas and matplotlib.

Below, we plot the distribution of a numeric field and a bar chart for group means. Update field `@id`s if necessary.

In [ ]:
# Example visualization
if dataframes:
    df = dataframes[preview_record_set_id]

    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(8,4))
        df[numeric_field_id].hist(bins=10)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    cat_cols = df.select_dtypes(exclude=['number']).columns.tolist()
    if numeric_cols and cat_cols:
        group_field_id = cat_cols[0]
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        plt.bar(grouped[group_field_id], grouped[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset via Croissant schema and explored its structure using entity `@id`s for record sets, fields, and columns. We extracted data, processed numeric and categorical fields, and visualized distributions and group statistics.

Key findings:
- The dataset contains structured tables on clinical features, hormone levels, imaging, and genetic variants for three female patients with NC-21OHD-associated infertility.
- Data is limited in scale but suitable for case-based phenotypic analysis and academic investigation.

**Note**: For deeper analyses, refer to the actual record set and field `@id`s enumerated in section 2, and always cite entities and columns using their `@id` in accordance with Croissant and FAIR^2 best practices.